# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library.

### Dataset Source
The dataset is described by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and discover the dataset using `mlcroissant`. We'll start by loading the metadata, printing basic summary information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
md = dataset.metadata

print(f"Dataset title: {md.name}")
print(f"Description: {md.description}")
print(f"Published: {md.datePublished}")
print(f"Keywords: {getattr(md, 'keywords', None)}")
print(f"Identifier (DOI): {getattr(md, 'identifier', None)}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We use the Croissant schema structure to inspect record sets, fields, and columns using their `@id` attributes as required for reference. This also helps discover the structure before extraction.

In [ ]:
# Enumerate all available record sets and their field @ids

def print_recordset_structure(ds):
    print("Available record sets and contained fields/columns (by '@id'):")
    for rs in ds.metadata.recordSets:
        print(f"- Record Set '@id': {rs.id if hasattr(rs,'id') else getattr(rs,'@id','<missing>')}")
        print(f"  Name: {getattr(rs, 'name', None)}")
        # List fields and columns
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields/Columns:")
            for field in rs.fields:
                print(f"    - Field '@id': {getattr(field, 'id', getattr(field, '@id', '<missing>'))}, name: {getattr(field, 'name', getattr(field, 'label', None))}")
        elif hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - Column '@id': {getattr(col, 'id', getattr(col, '@id', '<missing>'))}, name: {getattr(col, 'name', getattr(col, 'label', None))}")
        print()

print_recordset_structure(dataset)

## 3. Data Extraction

Load data from a specific record set into a DataFrame. Use the record set and field `@id`s identified above.

**Note**: Replace `<record_set_id>` with the actual `@id` as needed, using only `@id` values for referencing. Below, we extract all record sets into DataFrames.

In [ ]:
# Gather all record set @ids
record_sets_ids = [getattr(rs, 'id', getattr(rs, '@id', None)) for rs in dataset.metadata.recordSets]
print(f"Discovered record set @ids: {record_sets_ids}\n")

dataframes = dict()

# Load records for each record set
for record_set_id in record_sets_ids:
    print(f"\nLoading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping data. The EDA below shows how to work with fields using their `@id` as required by the Croissant schema.

> **Note:** Update field `@id` values to match the actual dataset if you wish to analyze a different numeric field.

In [ ]:
# For demonstration, pick the first available record set (update as needed for your analysis)
if record_sets_ids:
    record_set_id = record_sets_ids[0]
    df = dataframes[record_set_id]

    # Identify numeric fields by examining column names and dtypes
    print("Available columns:", df.columns.tolist())
    print("\nField types:")
    print(df.dtypes)
    
    # Try common names for protein mass or abundance if present
    numeric_field = None
    for c in df.columns:
        if 'abund' in c.lower() or 'mw' in c.lower() or 'weight' in c.lower() or df[c].dtype in [float, int]:
            numeric_field = c
            print(f"Selected numeric field: {numeric_field}")
            break
    if numeric_field is None:
        print("No numeric fields found for filtering. Please update numeric_field variable manually.")
    else:
        # Replace threshold as appropriate for column
        threshold = df[numeric_field].median() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, col_norm]].head())

        # Try grouping by a non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No record_set_id available; check schema loading above.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. You can use any DataFrame created above and plot chosen fields. Below, a histogram is shown for the normalized numeric field, with a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets_ids and 'filtered_df' in locals() and numeric_field:
    col_norm = f"{numeric_field}_normalized"
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[col_norm], kde=True)
    plt.title(f"Distribution of normalized {numeric_field}")
    plt.xlabel(f"{numeric_field} (normalized)")
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=col_norm)
        plt.title(f"Normalized {numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field data for visualization.")

## 6. Conclusion

- Successfully loaded **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** via the Croissant schema using `mlcroissant`.
- Inspected available record sets and their field `@id`s.
- Loaded tabular data into DataFrames, demonstrated filtering, normalization, grouping, and plotting with clear, robust reference to field and record set `@id`s as per the Croissant standard.
- You can further adapt the notebook to focus on specific protein attributes or expand to cross-record-set analysis as needed.

For more details, consult [Croissant documentation](https://mlcommons.github.io/croissant/) and [mlcroissant Python docs](https://mlcroissant.readthedocs.io/en/latest/).